# Task 4: Model Evaluation
Test models on test data, compute metrics, and check for overfitting/underfitting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score, roc_curve)
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv('cardio_cleaned.csv')
X = df.drop('cardio', axis=1)
y = df['cardio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = joblib.load('scaler.pkl')
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Linear Regression': joblib.load('model_linear_regression.pkl'),
    'Logistic Regression': joblib.load('model_logistic_regression.pkl'),
    'SVM': joblib.load('model_svm.pkl'),
    'KNN': joblib.load('model_knn.pkl'),
    'Decision Tree': joblib.load('model_decision_tree.pkl'),
    'Random Forest': joblib.load('model_random_forest.pkl')
}
print(f"Loaded {len(models)} models.")

## 4.1 Compute Evaluation Metrics

In [ ]:
results = []

for name, model in models.items():
    if name == 'Linear Regression':
        y_pred = (model.predict(X_test_scaled) >= 0.5).astype(int)
        y_train_pred = (model.predict(X_train_scaled) >= 0.5).astype(int)
    else:
        y_pred = model.predict(X_test_scaled)
        y_train_pred = model.predict(X_train_scaled)

    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_pred)

    results.append({
        'Model': name,
        'Train Accuracy': round(train_acc, 4),
        'Test Accuracy': round(test_acc, 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1 Score': round(f1_score(y_test, y_pred), 4),
        'Overfit Gap': round(train_acc - test_acc, 4)
    })

results_df = pd.DataFrame(results).sort_values('Test Accuracy', ascending=False).reset_index(drop=True)
results_df

## 4.2 Overfitting / Underfitting Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(len(results_df))
width = 0.35

bars1 = ax.bar(x_pos - width/2, results_df['Train Accuracy'], width,
               label='Train', color='#3498db', alpha=0.85)
bars2 = ax.bar(x_pos + width/2, results_df['Test Accuracy'], width,
               label='Test', color='#e74c3c', alpha=0.85)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Train vs Test Accuracy — Overfitting Check', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(results_df['Model'], rotation=25, ha='right')
ax.legend()
ax.set_ylim(0.6, 1.0)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\nOverfitting/Underfitting Analysis:")
for _, row in results_df.iterrows():
    gap = row['Overfit Gap']
    status = "✓ Good Fit" if abs(gap) < 0.03 else ("⚠ Overfitting" if gap > 0.03 else "⚠ Underfitting")
    print(f"  {row['Model']:25s} | Gap: {gap:+.4f} | {status}")

## 4.3 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for idx, (name, model) in enumerate(models.items()):
    row, col = idx // 3, idx % 3
    if name == 'Linear Regression':
        y_pred = (model.predict(X_test_scaled) >= 0.5).astype(int)
    else:
        y_pred = model.predict(X_test_scaled)

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[row][col],
                xticklabels=['No Disease', 'Disease'],
                yticklabels=['No Disease', 'Disease'])
    axes[row][col].set_title(f'{name}', fontweight='bold')
    axes[row][col].set_ylabel('Actual')
    axes[row][col].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

## 4.4 ROC Curves

In [ ]:
plt.figure(figsize=(10, 8))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

for idx, (name, model) in enumerate(models.items()):
    if name == 'Linear Regression':
        y_scores = model.predict(X_test_scaled)
    elif hasattr(model, 'predict_proba'):
        y_scores = model.predict_proba(X_test_scaled)[:, 1]
    else:
        continue

    fpr, tpr, _ = roc_curve(y_test, y_scores)
    auc = roc_auc_score(y_test, y_scores)
    plt.plot(fpr, tpr, color=colors[idx], linewidth=2, label=f'{name} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.5)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()